## Подготовка

In [1]:
import pandas as pd
import nltk
import nltk.corpus
import nltk.tokenize
import re
import pymorphy2
import locale
import numpy as np
import gensim
from string import punctuation
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec, FastText
from gensim.models.fasttext import load_facebook_model

from rus_preprocessing_udpipe import num_replace, clean_token, clean_lemma, list_replace, unify_sym, process
import sys
import os
import wget
import re
from ufal.udpipe import Model, Pipeline


Loading the model...
Processing input...


In [2]:
tqdm.pandas()

In [3]:
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\den26\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\den26\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
negative_file = "negative.csv"
positive_file = "positive.csv"
header_names = ["id", "tdate", "tmane",'ttext', 'ttype', 'trep', 'trtv','tfav', 'tstcount', 'tfol', 'tfrien', 'listcount']

negative_df = pd.read_csv(negative_file, delimiter=";",header=None, names = header_names)
positive_df = pd.read_csv(positive_file, delimiter=";",header=None, names = header_names)
negative_df = negative_df.sample(5000, random_state = 42)
positive_df = positive_df.sample(5000, random_state = 42)
df = pd.concat([negative_df, positive_df])

In [5]:
dictionary_file = "words_all_full_rating.csv"

dictionary = pd.read_csv(dictionary_file, delimiter = ";", encoding = "windows-1251")
dictionary["mean"] = dictionary["mean"].apply(lambda x: x.replace(',', '.'))
dictionary["dispersion"] = dictionary["dispersion"].apply(lambda x: x.replace(',', '.'))

In [7]:
# punctuations = set(punctuation)
# punctuations.update(['``','...',"''",'«','»','…','”','”','“','-','–','..'])
stopwords = nltk.corpus.stopwords.words("russian")
stopwords.remove("не")
stopwords.remove("нет")
stopwords.remove("ни")
# хорошо, лучше, нельзя
morph = pymorphy2.MorphAnalyzer()

def preprocess(text):
    text = re.sub("[^а-яА-ЯёЁ ]", ' ', text)
    text_tokenized = nltk.word_tokenize(text)
    text_tokenized = [x.lower() for x in text_tokenized if x not in stopwords]
    text_tokenized = [morph.parse(word)[0].normal_form for word in text_tokenized]
    return text_tokenized

In [52]:
" ".join(stopwords)

'и в во что он на я с со как а то все она так его но да ты к у же вы за бы по только ее мне было вот от меня еще о из ему теперь когда даже ну вдруг ли если уже или быть был него до вас нибудь опять уж вам ведь там потом себя ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем всех никогда можно при наконец два об другой хоть после над больше тот через эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой перед иногда лучше чуть том нельзя такой им более всегда конечно всю между'

In [8]:
df["ttext_preprocessed"] = df["ttext"].progress_apply(preprocess)

100%|███████████████████████████████████| 10000/10000 [00:10<00:00, 985.80it/s]


## LinisCrowd 2015

In [9]:
def apply_linis(text_tokenized):
    text_transformed = [float(dictionary[dictionary["Words"] == word]["mean"]) 
                             for word in text_tokenized if word in dictionary["Words"].array]
    if len(text_transformed) == 0:
        return [0.0]
    
    return text_transformed

In [10]:
linis_df = df[["ttext_preprocessed", "ttype"]]
linis_df["ttext_vectors"] = linis_df["ttext_preprocessed"].progress_apply(apply_linis)

  0%|                                                | 0/10000 [00:00<?, ?it/s]C:\Users\den26\AppData\Local\Temp/ipykernel_17404/2501804651.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  text_transformed = [float(dictionary[dictionary["Words"] == word]["mean"])
100%|███████████████████████████████████| 10000/10000 [00:40<00:00, 243.99it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/929231080.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linis_df["ttext_vectors"] = linis_df["ttext_preprocessed"].progress_apply(apply_linis)


In [11]:
ttext_vectors = linis_df["ttext_vectors"]
linis_df["mean"] = ttext_vectors.apply(np.mean)
linis_df["max"] = ttext_vectors.apply(np.max)
linis_df["min"] = ttext_vectors.apply(np.min)
linis_df["sum"] = ttext_vectors.apply(np.sum)
linis_df["neg_count"] = ttext_vectors.apply(lambda x: np.sum([1 if number < 0 else 0 for number in x]))
linis_df["pos_count"] = ttext_vectors.apply(lambda x: np.sum([1 if number > 0 else 0 for number in x]))

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1937367665.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linis_df["mean"] = ttext_vectors.apply(np.mean)
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1937367665.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  linis_df["max"] = ttext_vectors.apply(np.max)
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1937367665.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_in

In [12]:
linis_df.to_csv("linis.csv")

In [13]:
X = linis_df[["mean", "max", "min", "sum", "neg_count", "pos_count"]]
y = linis_df[["ttype"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [14]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

c:\users\den26\appdata\local\programs\python\python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

          -1     0.5479    0.7552    0.6350      1250
           1     0.6062    0.3768    0.4647      1250

    accuracy                         0.5660      2500
   macro avg     0.5770    0.5660    0.5499      2500
weighted avg     0.5770    0.5660    0.5499      2500



## Pymorphy2

In [15]:
def apply_pymorphy(text_tokenized):
    return [morph.parse(word)[0].tag.POS for word in text_tokenized]

def count_value(text, word):
    count = 0
    for w in text:
        if word == w:
            count += 1
    
    return count

In [16]:
morph_df = df[["ttext_preprocessed", "ttype"]]

In [17]:
morph_df["ttext_vectors"] = morph_df["ttext_preprocessed"].progress_apply(apply_pymorphy)

100%|██████████████████████████████████| 10000/10000 [00:08<00:00, 1166.10it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/3774293169.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  morph_df["ttext_vectors"] = morph_df["ttext_preprocessed"].progress_apply(apply_pymorphy)


In [18]:
parts_of_speech = set()
morph_df["ttext_vectors"].apply(lambda x: [parts_of_speech.add(word) for word in x])
parts_of_speech.remove(None)

In [19]:
for part_of_speech in parts_of_speech:
    morph_df[part_of_speech] = morph_df["ttext_vectors"].apply(lambda x: count_value(x, part_of_speech) / max(len(x), 1))

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/944274497.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  morph_df[part_of_speech] = morph_df["ttext_vectors"].apply(lambda x: count_value(x, part_of_speech) / max(len(x), 1))
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/944274497.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  morph_df[part_of_speech] = morph_df["ttext_vectors"].apply(lambda x: count_value(x, part_of_speech) / max(len(x), 1))
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/94

In [20]:
morph_df.to_csv("pymorphy2.csv")

In [21]:
X = morph_df[list(parts_of_speech)]
y = morph_df[["ttype"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [22]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

c:\users\den26\appdata\local\programs\python\python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

          -1     0.5540    0.5416    0.5477      1250
           1     0.5516    0.5640    0.5578      1250

    accuracy                         0.5528      2500
   macro avg     0.5528    0.5528    0.5527      2500
weighted avg     0.5528    0.5528    0.5527      2500



## TF-IDF

In [23]:
tfidf_vectorizer = TfidfVectorizer(max_features = 1000)
tfidf_df = pd.DataFrame(tfidf_vectorizer.fit_transform(df["ttext_preprocessed"].apply(' '.join)).toarray())
tfidf_df.columns = tfidf_vectorizer.get_feature_names_out()
tfidf_df.index = df.index
tfidf_df["ttype"] = df["ttype"]

In [24]:
tfidf_df.to_csv("tfidf.csv")

In [25]:
X = tfidf_df[tfidf_vectorizer.get_feature_names_out()]
y = tfidf_df[["ttype"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [26]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

c:\users\den26\appdata\local\programs\python\python39\lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


              precision    recall  f1-score   support

          -1     0.6343    0.6576    0.6457      1250
           1     0.6445    0.6208    0.6324      1250

    accuracy                         0.6392      2500
   macro avg     0.6394    0.6392    0.6391      2500
weighted avg     0.6394    0.6392    0.6391      2500



## Word2Vec

In [27]:
ttext_preprocessed = df["ttext_preprocessed"]

In [28]:
word2vec = Word2Vec(sentences=ttext_preprocessed, vector_size=300, window=5, min_count=1, workers=4)
word2vec.build_vocab(ttext_preprocessed)
word2vec.train(ttext_preprocessed, total_examples=word2vec.corpus_count, epochs=300, report_delay=1)
word2vec.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1083997977.py:4: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  word2vec.init_sims(replace=True)


In [29]:
word2vec_df = ttext_preprocessed
word2vec_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: word2vec.wv[x]).array)
word2vec_df = word2vec_df["ttext_vectors"].progress_apply(lambda x: pd.Series(np.mean(x)))
word2vec_df["ttype"] = df["ttype"]

100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 8482.81it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1613646325.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word2vec_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: word2vec.wv[x]).array)
100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 8422.57it/s]


In [53]:
word2vec_df.to_csv("word2vec.csv")

In [30]:
X = word2vec_df[[i for i in range(300)]]
y = word2vec_df["ttype"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [31]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

              precision    recall  f1-score   support

          -1     0.6258    0.6248    0.6253      1250
           1     0.6254    0.6264    0.6259      1250

    accuracy                         0.6256      2500
   macro avg     0.6256    0.6256    0.6256      2500
weighted avg     0.6256    0.6256    0.6256      2500



## Word2Vec pretrained

In [32]:
udpipe_model_url = "https://rusvectores.org/static/models/udpipe_syntagrus.model"
udpipe_filename = udpipe_model_url.split("/")[-1]

if not os.path.isfile(udpipe_filename):
    wget.download(udpipe_model_url)

model = Model.load(udpipe_filename)
process_pipeline = Pipeline(
    model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu"
)

def process_text(text):
    res = unify_sym(text.strip())
    output = process(process_pipeline, text=res)
    return " ".join(output)

In [33]:
word2vec_pretrained_df = df[["ttext", "ttype"]]
word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext"].progress_apply(lambda x: re.sub("[^а-яА-ЯёЁ ]", ' ', x))
word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext_preprocessed"].progress_apply(process_text)
word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext_preprocessed"].progress_apply(lambda x: x.split())

100%|████████████████████████████████| 10000/10000 [00:00<00:00, 212673.49it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/2817615677.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext"].progress_apply(lambda x: re.sub("[^а-яА-ЯёЁ ]", ' ', x))
100%|███████████████████████████████████| 10000/10000 [00:59<00:00, 166.74it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/2817615677.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus

In [34]:
model = gensim.models.KeyedVectors.load_word2vec_format('word2vec_model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)
model.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/2829029209.py:2: DeprecationWarning: Call to deprecated `init_sims` (Use fill_norms() instead. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  model.init_sims(replace=True)


In [35]:
word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext_preprocessed"].apply(lambda x: [word for word in x if word in model])

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/3354765996.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word2vec_pretrained_df["ttext_preprocessed"] = word2vec_pretrained_df["ttext_preprocessed"].apply(lambda x: [word for word in x if word in model])


In [36]:
word2vec_pretrained_df["ttext_vectors"] = word2vec_pretrained_df["ttext_preprocessed"].progress_apply(lambda x: pd.Series(x).apply(lambda x: model[x] if x in model else [0] * 300).array)
word2vec_pretrained_df = word2vec_pretrained_df["ttext_vectors"].progress_apply(lambda x: pd.Series(np.mean(x)))
word2vec_pretrained_df["ttype"] = df["ttype"]

100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 8633.27it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/3536110383.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  word2vec_pretrained_df["ttext_vectors"] = word2vec_pretrained_df["ttext_preprocessed"].progress_apply(lambda x: pd.Series(x).apply(lambda x: model[x] if x in model else [0] * 300).array)
100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 8402.61it/s]


In [54]:
word2vec_pretrained_df.to_csv("word2vec_pretrained.csv")

In [37]:
X = word2vec_pretrained_df[[i for i in range(300)]]
y = word2vec_pretrained_df["ttype"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [38]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

              precision    recall  f1-score   support

          -1     0.6209    0.6288    0.6248      1250
           1     0.6240    0.6160    0.6200      1250

    accuracy                         0.6224      2500
   macro avg     0.6224    0.6224    0.6224      2500
weighted avg     0.6224    0.6224    0.6224      2500



## Fasttext

In [39]:
ttext_preprocessed = df["ttext_preprocessed"]

In [40]:
fasttext = FastText(vector_size=300, window=5, min_count=1, workers=4)
fasttext.build_vocab(corpus_iterable = ttext_preprocessed)
fasttext.train(ttext_preprocessed, total_examples=fasttext.corpus_count, epochs=10, report_delay=1)
fasttext.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/3947742650.py:4: DeprecationWarning: Call to deprecated `init_sims` (Gensim 4.0.0 implemented internal optimizations that make calls to init_sims() unnecessary. init_sims() is now obsoleted and will be completely removed in future versions. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  fasttext.init_sims(replace=True)


In [41]:
fasttext_df = ttext_preprocessed
fasttext_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: fasttext.wv[x]).array)
fasttext_df = fasttext_df["ttext_vectors"].progress_apply(lambda x: pd.Series(np.mean(x)))
fasttext_df["ttype"] = df["ttype"]

100%|██████████████████████████████████| 10000/10000 [00:01<00:00, 6924.38it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1930790224.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasttext_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: fasttext.wv[x]).array)
100%|██████████████████████████████████| 10000/10000 [00:06<00:00, 1537.53it/s]


In [55]:
fasttext_df.to_csv("fasttext.csv")

In [42]:
X = fasttext_df[[i for i in range(300)]]
y = fasttext_df["ttype"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [43]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

              precision    recall  f1-score   support

          -1     0.5648    0.5616    0.5632      1250
           1     0.5640    0.5672    0.5656      1250

    accuracy                         0.5644      2500
   macro avg     0.5644    0.5644    0.5644      2500
weighted avg     0.5644    0.5644    0.5644      2500



## Fasttext pretrained

In [44]:
ttext_preprocessed = df["ttext_preprocessed"]

In [45]:
model = gensim.models.fasttext.FastTextKeyedVectors.load('fasttext_model.model')
model.init_sims(replace=True)

C:\Users\den26\AppData\Local\Temp/ipykernel_17404/1605128550.py:2: DeprecationWarning: Call to deprecated `init_sims` (Use fill_norms() instead. See https://github.com/RaRe-Technologies/gensim/wiki/Migrating-from-Gensim-3.x-to-4).
  model.init_sims(replace=True)


In [47]:
fasttext_pretrained_df = ttext_preprocessed
fasttext_pretrained_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: model[x]).array)
fasttext_pretrained_df = fasttext_pretrained_df["ttext_vectors"].progress_apply(lambda x: pd.Series(np.mean(x)))
fasttext_pretrained_df["ttype"] = df["ttype"]

100%|███████████████████████████████████| 10000/10000 [00:36<00:00, 272.23it/s]
C:\Users\den26\AppData\Local\Temp/ipykernel_17404/834941804.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fasttext_pretrained_df["ttext_vectors"] = ttext_preprocessed.progress_apply(lambda x: pd.Series(x).apply(lambda x: model[x]).array)
100%|███████████████████████████████████| 10000/10000 [00:31<00:00, 322.21it/s]


In [48]:
fasttext_pretrained_df.to_csv("fasttext_pretrained.csv")

In [49]:
X = fasttext_pretrained_df[[i for i in range(300)]]
y = fasttext_pretrained_df["ttype"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, stratify = y, random_state = 42)

In [50]:
classifier = RandomForestClassifier(random_state = 42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
print(classification_report(y_test, y_pred, digits = 4))

              precision    recall  f1-score   support

          -1     0.6280    0.6536    0.6405      1250
           1     0.6389    0.6128    0.6256      1250

    accuracy                         0.6332      2500
   macro avg     0.6334    0.6332    0.6330      2500
weighted avg     0.6334    0.6332    0.6330      2500

